In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from music21 import converter, chord, note, stream

# ==========================================
# 1. Análisis de Notas por Sílaba (Melismas)
# ==========================================
def analizar_melismas(score):
    """
    Analiza la relación entre notas y sílabas en las voces cantadas.
    Calcula cuántas notas tienen sílabas asignadas (silábico) 
    y cuántas son extensiones melismáticas (sin sílaba nueva).
    """
    total_notas_vocales = 0
    total_silabas = 0
    
    # Extraer solo las partes que tienen texto (ignorando el Basso Continuo)
    for part in score.parts:
        notas = part.flatten().notes
        tiene_letra = any(n.lyric is not None for n in notas)
        
        if tiene_letra:
            for n in notas:
                # Contamos solo notas individuales, no acordes
                if isinstance(n, note.Note):
                    total_notas_vocales += 1
                    if n.lyric is not None:
                        total_silabas += 1
                        
    notas_melismaticas = total_notas_vocales - total_silabas
    
    # Prevenir división por cero
    if total_silabas == 0: return None
    
    return {
        'Notas_Silabicas': total_silabas,
        'Notas_Melismaticas': notas_melismaticas,
        'Ratio_Nota_por_Silaba': round(total_notas_vocales / total_silabas, 2)
    }

# ==========================================
# 2. Análisis de Disonancia Vertical
# ==========================================
def analizar_disonancias(score):
    """
    'Acordifica' la partitura (comprime todas las voces en un solo pentagrama de acordes).
    Evalúa cada momento vertical para determinar si es consonante o disonante
    según las reglas de la armonía tradicional.
    """
    # chordify() agrupa todas las notas que suenan al mismo tiempo
    acordes = score.chordify().flatten().getElementsByClass(chord.Chord)
    
    total_momentos = len(acordes)
    momentos_disonantes = 0
    momentos_consonantes = 0
    
    for c in acordes:
        if c.isConsonant():
            momentos_consonantes += 1
        else:
            momentos_disonantes += 1
            
    if total_momentos == 0: return None
            
    return {
        'Consonancias': momentos_consonantes,
        'Disonancias': momentos_disonantes,
        'Porcentaje_Disonancia': round((momentos_disonantes / total_momentos) * 100, 2)
    }

# ==========================================
# 3. Métrica Sugerida: Variación Rítmica
# ==========================================
def analizar_variacion_ritmica(score):
    """
    Cuenta cuántos tipos diferentes de duraciones rítmicas existen en la pieza.
    Una mayor variedad (corcheas, síncopas, fusas) suele indicar el estilo recitativo
    y la sprezzatura de la Seconda Pratica.
    """
    duraciones = []
    for n in score.flatten().notes:
        duraciones.append(n.quarterLength)
        
    # Cantidad de figuras rítmicas únicas usadas en la pieza
    variedad_ritmica = len(set(duraciones))
    
    return {'Figuras_Ritmicas_Unicas': variedad_ritmica}

# ==========================================
# 4. Función Principal de Procesamiento
# ==========================================
def procesar_pieza(ruta_xml, titulo, libro):
    """
    Carga el XML y ejecuta todos los análisis, devolviendo una fila de datos.
    """
    pieza = converter.parse(ruta_xml)
    
    datos_melismas = analizar_melismas(pieza)
    datos_disonancia = analizar_disonancias(pieza)
    datos_ritmo = analizar_variacion_ritmica(pieza)
    
    fila = {
        'Titulo': titulo,
        'Libro': libro,
        'Practica': 'Prima' if libro < 5 else 'Seconda'
    }
    
    if datos_melismas: fila.update(datos_melismas)
    if datos_disonancia: fila.update(datos_disonancia)
    if datos_ritmo: fila.update(datos_ritmo)
        
    return fila

# ==========================================
# 5. Visualización de Datos (Plots)
# ==========================================
def visualizar_datos(df):
    """
    Genera gráficos de barras y torta (pie charts) comparando 
    la Prima Pratica vs la Seconda Pratica.
    """
    # Agrupar datos por Práctica
    resumen = df.groupby('Practica').mean(numeric_only=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Análisis Simbólico: Prima vs Seconda Pratica en Monteverdi', fontsize=16)

    # 1. Bar Chart: Disonancia Promedio
    resumen['Porcentaje_Disonancia'].plot(kind='bar', ax=axes[0, 0], color=['#4C72B0', '#C44E52'])
    axes[0, 0].set_title('Promedio de Disonancia Vertical (%)')
    axes[0, 0].set_ylabel('% Disonancia')
    axes[0, 0].tick_params(axis='x', rotation=0)

    # 2. Bar Chart: Variedad Rítmica
    resumen['Figuras_Ritmicas_Unicas'].plot(kind='bar', ax=axes[0, 1], color=['#55A868', '#DD8452'])
    axes[0, 1].set_title('Variedad Rítmica (Figuras únicas promedio)')
    axes[0, 1].set_ylabel('Cantidad de figuras rítmicas')
    axes[0, 1].tick_params(axis='x', rotation=0)

    # 3. Pie Chart: Melismas vs Silábico (Ejemplo global)
    totales = df[['Notas_Silabicas', 'Notas_Melismaticas']].sum()
    axes[1, 0].pie(totales, labels=['Silábicas (con texto)', 'Melismáticas (ligadas)'], autopct='%1.1f%%', startangle=90)
    axes[1, 0].set_title('Distribución Global de Textura Vocal')

    # 4. Bar Chart: Ratio Notas por Sílaba
    resumen['Ratio_Nota_por_Silaba'].plot(kind='bar', ax=axes[1, 1], color=['#8172B2', '#64B5CD'])
    axes[1, 1].set_title('Ratio Promedio: Notas por Sílaba')
    axes[1, 1].set_ylabel('Notas / Sílabas')
    axes[1, 1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()

# --- EJEMPLO DE USO (Simulado) ---
# df = pd.DataFrame([
#    procesar_pieza("ruta/a/libro4_1.xml", "Sfogava con le stelle", 4),
#    procesar_pieza("ruta/a/libro5_1.xml", "Cruda Amarilli", 5)
# ])
# visualizar_datos(df)